# Six-Model Foundation-Model MD Analysis

Workflow template for visualizing regenerated bulk MD trajectories for SevenNet-0, MatterSim, UPET, MACE-MH, UMA, and NequIP-OAM-L. Run `scripts/run_bulk_all_models.sh` before executing the analysis cells.


In [ ]:

from pathlib import Path
from collections import defaultdict
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde
from scipy.signal import savgol_filter
from ase.io.trajectory import Trajectory

ROOT = Path.cwd()
OLD_BULK_RESULTS = ROOT / "results"
NEW_BULK_RESULTS = ROOT / "results_bulk"
PERF_DIR = ROOT / "results" / "computational_performance"

# plt.rcParams.update({
#     "font.size": 11,
#     "axes.spines.top": False,
#     "axes.spines.right": False,
# })


In [ ]:
import json
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from collections import defaultdict
from ase.io.trajectory import Trajectory

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "legend.fontsize": 9,  "xtick.labelsize": 10, "ytick.labelsize": 10,
    "figure.dpi": 120,
})


In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.legend as lg
import matplotlib.ticker as ticker
from scipy import interpolate
from matplotlib.ticker import LogFormatterSciNotation 
import matplotlib as mpl
import os, sys
import pandas as pd 
import matplotlib.style
import matplotlib.patches as patches


import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times", "Nimbus Roman", "DejaVu Serif"],
})


mpl.rcParams['font.size'] = 23
mpl.rcParams["font.weight"] = "bold"
# mpl.rcParams['font.family'] = 'Georgia'
mpl.rcParams['legend.fontsize'] = 23
mpl.rcParams['axes.linewidth'] = 3.0
mpl.rcParams['lines.linewidth'] = 3.0
mpl.rcParams['axes.labelweight'] = "bold"
mpl.rcParams['axes.titleweight'] = "bold"
mpl.rcParams['xtick.major.size'] = 8
mpl.rcParams['xtick.top']=True
mpl.rcParams['ytick.right']=True
mpl.rcParams['xtick.minor.size'] = 4
mpl.rcParams['xtick.major.width'] = 2
mpl.rcParams['xtick.minor.width'] = 1.5
mpl.rcParams['xtick.direction'] = "in"
mpl.rcParams['ytick.major.size'] = 8
mpl.rcParams['ytick.minor.size'] = 4
mpl.rcParams['ytick.major.width'] = 2
mpl.rcParams['ytick.minor.width'] = 1
mpl.rcParams['ytick.direction'] = "in"
mpl.rcParams['xtick.major.top'] = True
mpl.rcParams['xtick.minor.top'] = True
mpl.rcParams['ytick.major.right'] = True
mpl.rcParams['ytick.minor.right'] = True

In [ ]:

MODELS = ["sevennet", "mattersim", "upet", "mace_mh", "uma", "nequip_oam_l"]
MODEL_LABELS = {
    "sevennet": "SevenNet-0",
    "mattersim": "MatterSim",
    "upet": "UPET",
    "mace_mh": "MACE-MH",
    "uma": "UMA",
    "nequip_oam_l": "NequIP-OAM-L",
}
MODEL_COLORS = {
    "sevennet": "#1f77b4",
    "mattersim": "#2ca02c",
    "upet": "#ff7f0e",
    "mace_mh": "#d62728",
    "uma": "#9467bd",
    "nequip_oam_l": "#17becf",
}
MODEL_LS = {
    "sevennet": "-",
    "mattersim": "--",
    "upet": "-.",
    "mace_mh": ":",
    "uma": "-",
    "nequip_oam_l": "--",
}
MODEL_MARKERS = {
    "sevennet": "o",
    "mattersim": "s",
    "upet": "^",
    "mace_mh": "D",
    "uma": "P",
    "nequip_oam_l": "X",
}
MODEL_TRAJ_DIRS = {
    "sevennet": OLD_BULK_RESULTS / "sevennet",
    "mattersim": OLD_BULK_RESULTS / "mattersim",
    "upet": OLD_BULK_RESULTS / "upet",
    "mace_mh": NEW_BULK_RESULTS / "mace_mh1",
    "uma": NEW_BULK_RESULTS / "uma",
    "nequip_oam_l": NEW_BULK_RESULTS / "nequip_oam_l",
}
MODEL_TIME_KEYS = {
    "sevennet": "sevennet",
    "mattersim": "mattersim",
    "upet": "upet",
    "mace_mh": "mace_mh1",
    "uma": "uma",
    "nequip_oam_l": "nequip_oam_l",
}

STRUCTURES = ["Mg2SiO4_balanced", "Fe2SiO4_balanced", "TiFeO3_balanced", "CaAl2Si2O8_balanced"]
SLABELS = {
    "Mg2SiO4_balanced": "Mg₂SiO₄ (Forsterite)",
    "Fe2SiO4_balanced": "Fe₂SiO₄ (Fayalite)",
    "TiFeO3_balanced": "TiFeO₃ (Ilmenite)",
    "CaAl2Si2O8_balanced": "CaAl₂Si₂O₈ (Anorthite)",
}
SHORT_LABELS = ["Mg₂SiO₄", "Fe₂SiO₄", "TiFeO₃", "CaAl₂Si₂O₈"]

BOND_PAIRS = {
    "Mg2SiO4_balanced": [("Si", "O", 2.0, (1.4, 2.0)), ("Mg", "O", 2.7, (1.8, 2.7))],
    "Fe2SiO4_balanced": [("Si", "O", 2.0, (1.4, 2.0)), ("Fe", "O", 2.7, (1.8, 2.7))],
    "TiFeO3_balanced": [("Ti", "O", 2.4, (1.6, 2.4)), ("Fe", "O", 2.7, (1.8, 2.7))],
    "CaAl2Si2O8_balanced": [("Si", "O", 2.0, (1.4, 2.0)), ("Al", "O", 2.2, (1.6, 2.2)), ("Ca", "O", 3.0, (2.0, 3.0))],
}
ANGLE_TRIPLETS = {
    "Mg2SiO4_balanced": [("O", "Si", "O", 2.0, 2.0, "O-Si-O", (80, 140)), ("Mg", "O", "Mg", 2.7, 2.7, "Mg-O-Mg", (80, 140)), ("Mg", "O", "Si", 2.7, 2.0, "Mg-O-Si", (80, 160))],
    "Fe2SiO4_balanced": [("O", "Si", "O", 2.0, 2.0, "O-Si-O", (80, 140)), ("Fe", "O", "Fe", 2.7, 2.7, "Fe-O-Fe", (80, 140)), ("Fe", "O", "Si", 2.7, 2.0, "Fe-O-Si", (80, 160))],
    "TiFeO3_balanced": [("O", "Ti", "O", 2.4, 2.4, "O-Ti-O", (60, 180)), ("O", "Fe", "O", 2.7, 2.7, "O-Fe-O", (60, 180)), ("Ti", "O", "Fe", 2.4, 2.7, "Ti-O-Fe", (80, 160))],
    "CaAl2Si2O8_balanced": [("O", "Si", "O", 2.0, 2.0, "O-Si-O", (80, 140)), ("O", "Al", "O", 2.2, 2.2, "O-Al-O", (80, 140)), ("Si", "O", "Al", 2.0, 2.2, "Si-O-Al", (100, 180))],
}
GR_PAIRS = {
    "Mg2SiO4_balanced": [("Si", "O"), ("Mg", "O"), ("O", "O")],
    "Fe2SiO4_balanced": [("Si", "O"), ("Fe", "O"), ("O", "O")],
    "TiFeO3_balanced": [("Ti", "O"), ("Fe", "O"), ("O", "O")],
    "CaAl2Si2O8_balanced": [("Si", "O"), ("Al", "O"), ("Ca", "O"), ("O", "O")],
}


In [ ]:

def traj_path(model, struct):
    return MODEL_TRAJ_DIRS[model] / struct / "md.traj"


def log_path(model, struct):
    return MODEL_TRAJ_DIRS[model] / struct / "md.log"


def load_timing_details():
    rows = []
    for model in MODELS:
        for struct in STRUCTURES:
            p = MODEL_TRAJ_DIRS[model] / struct / "simulation_details.json"
            if not p.exists():
                continue
            info = json.loads(p.read_text())
            rows.append({
                "model": model,
                "model_label": MODEL_LABELS[model],
                "structure": struct,
                "mineral": SLABELS[struct],
                "n_atoms": info.get("num_atoms", np.nan),
                "n_steps": info.get("num_steps", 1000),
                "simulation_time_s": info.get("simulation_time_s", np.nan),
                "time_per_step_ms": info.get("time_per_step_ms", np.nan),
            })
    return pd.DataFrame(rows)


def load_md_log(model, struct):
    path = log_path(model, struct)
    if not path.exists():
        return None
    data = defaultdict(list)
    with path.open() as fh:
        for line in fh:
            s = line.strip()
            if not s or s.startswith("Time"):
                continue
            parts = s.split()
            if len(parts) == 5:
                try:
                    data["time"].append(float(parts[0]))
                    data["Etot"].append(float(parts[1]))
                    data["Epot"].append(float(parts[2]))
                    data["Ekin"].append(float(parts[3]))
                    data["T"].append(float(parts[4]))
                except ValueError:
                    pass
    return {k: np.asarray(v) for k, v in data.items()}


def _mic_vectors(pos_from, pos_to, cell):
    cell_inv = np.linalg.inv(cell.T)
    dv = pos_to - pos_from
    df = dv @ cell_inv.T
    df -= np.round(df)
    return df @ cell.T


def get_bond_distances(model, struct, sp_a, sp_b, cutoff, stride=10):
    path = traj_path(model, struct)
    if not path.exists():
        return np.array([])
    traj = Trajectory(str(path))
    all_dists = []
    for frame_i in range(0, len(traj), stride):
        atoms = traj[frame_i]
        pos = atoms.get_positions()
        syms = np.asarray(atoms.get_chemical_symbols())
        idx_a = np.where(syms == sp_a)[0]
        idx_b = np.where(syms == sp_b)[0]
        if len(idx_a) == 0 or len(idx_b) == 0:
            return np.array([])
        for pa in pos[idx_a]:
            dr = _mic_vectors(pa, pos[idx_b], atoms.get_cell())
            dist = np.sqrt((dr ** 2).sum(axis=1))
            if sp_a == sp_b:
                dist = dist[dist > 1e-3]
            all_dists.append(dist[dist < cutoff])
    return np.concatenate(all_dists) if all_dists else np.array([])


def get_bond_angles(model, struct, sp_end1, sp_center, sp_end2, cut1, cut2, stride=10):
    path = traj_path(model, struct)
    if not path.exists():
        return np.array([])
    traj = Trajectory(str(path))
    all_angles = []
    same = sp_end1 == sp_end2
    for frame_i in range(0, len(traj), stride):
        atoms = traj[frame_i]
        pos = atoms.get_positions()
        syms = np.asarray(atoms.get_chemical_symbols())
        cell = atoms.get_cell()
        idx_c = np.where(syms == sp_center)[0]
        idx_e1 = np.where(syms == sp_end1)[0]
        idx_e2 = np.where(syms == sp_end2)[0]
        if len(idx_c) == 0 or len(idx_e1) == 0 or len(idx_e2) == 0:
            return np.array([])
        for ic in idx_c:
            pc = pos[ic]
            dr1 = _mic_vectors(pc, pos[idx_e1], cell)
            d1 = np.sqrt((dr1 ** 2).sum(axis=1))
            dr2 = _mic_vectors(pc, pos[idx_e2], cell)
            d2 = np.sqrt((dr2 ** 2).sum(axis=1))
            mask1 = (d1 > 1e-3) & (d1 < cut1)
            mask2 = (d2 > 1e-3) & (d2 < cut2)
            if not mask1.any() or not mask2.any():
                continue
            n1 = dr1[mask1] / d1[mask1, None]
            n2 = dr2[mask2] / d2[mask2, None]
            if same:
                dots = n1 @ n1.T
                iu, ju = np.triu_indices(len(n1), k=1)
                cos_vals = np.clip(dots[iu, ju], -1, 1)
            else:
                cos_vals = np.clip((n1 @ n2.T).ravel(), -1, 1)
            all_angles.append(np.degrees(np.arccos(cos_vals)))
    return np.concatenate(all_angles) if all_angles else np.array([])


def compute_gr(model, struct, sp_a, sp_b, rmax=6.0, nbins=200, stride=10):
    path = traj_path(model, struct)
    if not path.exists():
        return None, None
    traj = Trajectory(str(path))
    start = len(traj) // 2
    bins = np.linspace(0, rmax, nbins + 1)
    centers = 0.5 * (bins[:-1] + bins[1:])
    hist = np.zeros(nbins)
    vol_sum = na_sum = nb_sum = n_frames = 0.0
    for frame_i in range(start, len(traj), stride):
        atoms = traj[frame_i]
        if atoms.get_volume() < 1.0:
            continue
        pos = atoms.get_positions()
        syms = np.asarray(atoms.get_chemical_symbols())
        idx_a = np.where(syms == sp_a)[0]
        idx_b = np.where(syms == sp_b)[0]
        if len(idx_a) == 0 or len(idx_b) == 0:
            return None, None
        vol_sum += atoms.get_volume()
        na_sum += len(idx_a)
        nb_sum += len(idx_b)
        n_frames += 1
        for pa in pos[idx_a]:
            dr = _mic_vectors(pa, pos[idx_b], atoms.get_cell())
            dist = np.sqrt((dr ** 2).sum(axis=1))
            if sp_a == sp_b:
                dist = dist[dist > 1e-3]
            h, _ = np.histogram(dist[dist < rmax], bins=bins)
            hist += h
    if n_frames == 0:
        return None, None
    rho_b = (nb_sum / n_frames) / (vol_sum / n_frames)
    shell_v = (4.0 / 3.0) * np.pi * (bins[1:] ** 3 - bins[:-1] ** 3)
    norm = (na_sum / n_frames) * n_frames * rho_b * shell_v
    if sp_a == sp_b:
        norm *= 0.5
    return centers, hist / norm

timing = load_timing_details()
logs = {(m, s): load_md_log(m, s) for m in MODELS for s in STRUCTURES}
missing = [(m, s) for m in MODELS for s in STRUCTURES if not traj_path(m, s).exists()]
print(f"Loaded timing rows: {len(timing)}")
print(f"Missing trajectories: {missing}")


## Computing Time Comparison

## MD Dynamics

In [ ]:
fig, axes = plt.subplots(1, len(STRUCTURES), figsize=(17, 4), sharey=True)

for ax, struct in zip(axes, STRUCTURES):
    for model in MODELS:
        d = logs.get((model, struct))
        if d is None or len(d.get("T", [])) == 0:
            continue
        
        # Swapped ax.plot for ax.scatter
        ax.scatter(d["time"] * 1000, d["T"], marker="x", s=2, alpha=0.82,
                   c=MODEL_COLORS[model], label=MODEL_LABELS[model])
         
    ax.axhline(300, color="0.5", ls="--", lw=0.8)
    ax.set_title(SLABELS[struct].split()[0])
    ax.set_xlabel("Time (fs)")
    ax.grid(alpha=0.2)

axes[0].set_ylabel("Temperature (K)")
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
# plt.show()

plt.savefig('fig2_temperature.png', dpi=200, bbox_inches='tight')

## Bond Distance Distributions

In [ ]:

bond_dist_data = {}
for struct in STRUCTURES:
    for sp_a, sp_b, cutoff, _ in BOND_PAIRS[struct]:
        for model in MODELS:
            arr = get_bond_distances(model, struct, sp_a, sp_b, cutoff, stride=10)
            bond_dist_data[(model, struct, sp_a, sp_b)] = arr
            print(f"{MODEL_LABELS[model]:14s} {struct:24s} {sp_a}-{sp_b}: {len(arr)} samples")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde

# 1. Setup the Figure
num_rows = len(STRUCTURES)
fig = plt.figure(figsize=(12, 4 * num_rows))
gs = gridspec.GridSpec(num_rows, 6, hspace=0.4, wspace=0.3)

for i, struct in enumerate(STRUCTURES):
    pairs = BOND_PAIRS.get(struct, [])
    n_pairs = len(pairs)
    
    if n_pairs == 0:
        continue

    # Asymmetrical Layout Logic
    if n_pairs == 3:
        width = 2
    else:
        width = 6 // n_pairs 

    for j in range(n_pairs):
        start_col = j * width
        ax = fig.add_subplot(gs[i, start_col : start_col + width])
        
        sp_a, sp_b, cutoff, xrange = pairs[j]
        x_smooth = np.linspace(xrange[0], xrange[1], 300)

        # Plotting each model
        for model in MODELS:
            arr = bond_dist_data.get((model, struct, sp_a, sp_b), np.array([]))
            
            if arr.size < 3:
                continue
                
            kde = gaussian_kde(arr, bw_method=0.18)
            y_smooth = kde.evaluate(x_smooth)
            
            ax.plot(x_smooth, y_smooth, color=MODEL_COLORS[model],
                    ls=MODEL_LS[model], lw=2.0, label=MODEL_LABELS[model])
            ax.fill_between(x_smooth, y_smooth, color=MODEL_COLORS[model], alpha=0.08)

        # --- NEW LEGEND LOGIC ---
        # Only add the legend to the very first plot (i=0, j=0)
        if i == 0 and j == 0:
            ax.legend(fontsize=9, frameon=False, loc='upper right')
# The code snippet you provided is attempting to create a global legend for the entire figure.

        # Refined Visualization
        ax.set_title(f"{sp_a}–{sp_b} Bond", fontsize=11, fontweight='bold', pad=8)
        ax.set_xlabel("Distance (Å)", fontsize=9)
        
        if j == 0:
            ax.set_ylabel(f"{SLABELS[struct]}", fontsize=12, fontweight='bold')
        
        ax.set_xlim(xrange[0], xrange[1])
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.15, linestyle='--')

# Global title and tight layout
# fig.suptitle("Radial Distribution Function Comparison", fontsize=15, fontweight='bold', y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('fig3_bond_distances.png', dpi=200, bbox_inches='tight')

## Bond Angle Distributions

In [ ]:

angle_data = {}
for struct in STRUCTURES:
    for sp_e1, sp_c, sp_e2, cut1, cut2, label, _ in ANGLE_TRIPLETS[struct]:
        for model in MODELS:
            arr = get_bond_angles(model, struct, sp_e1, sp_c, sp_e2, cut1, cut2, stride=10)
            angle_data[(model, struct, label)] = arr
            print(f"{MODEL_LABELS[model]:14s} {struct:24s} {label}: {len(arr)} samples")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde

# 1. Setup the Figure
num_rows = len(STRUCTURES)
# Keep height consistent with the bond plots for a matching set
fig = plt.figure(figsize=(24, 3.8 * num_rows))
gs = gridspec.GridSpec(num_rows, 6, hspace=0.45, wspace=0.3)

for i, struct in enumerate(STRUCTURES):
    triplets = ANGLE_TRIPLETS.get(struct, [])
    n_ang = len(triplets)
    
    if n_ang == 0:
        continue

    # Asymmetrical Layout Logic (LCM of 2 and 3 is 6)
    if n_ang == 3:
        width = 2
    else:
        # Default to 2 columns (6/2=3)
        width = 6 // n_ang 

    for j in range(n_ang):
        start_col = j * width
        ax = fig.add_subplot(gs[i, start_col : start_col + width])
        
        sp_e1, sp_c, sp_e2, cut1, cut2, label, xrange = triplets[j]
        x_smooth = np.linspace(xrange[0], xrange[1], 400)

        # # 1. Plot Reference Line (Tetrahedral 109.47°)
        # if "O" in sp_e1 and "O" in sp_e2 and sp_e1 == sp_e2:
        #     ax.axvline(109.47, color="red", ls="--", lw=1.2, alpha=0.5,
        #                label="109.47°", zorder=1)

        # 2. Plot Gaussian KDE for each model
        for model in MODELS:
            arr = angle_data.get((model, struct, label), np.array([]))
            
            if arr.size < 5:
                continue
                
            # Angles are usually peakier, so 0.15 is a good bandwidth
            kde = gaussian_kde(arr, bw_method=0.15)
            y_smooth = kde.evaluate(x_smooth)
            
            ax.plot(x_smooth, y_smooth, color=MODEL_COLORS[model],
                    ls=MODEL_LS[model], lw=2.0, label=MODEL_LABELS[model], zorder=2)
            ax.fill_between(x_smooth, y_smooth, color=MODEL_COLORS[model], alpha=0.08, zorder=2)

        # --- Legend on first plot only ---
        if i == 0 and j == 0:
            ax.legend(fontsize=8, frameon=False, loc='upper right')

        # Formatting
        ax.set_title(f"{label} Distribution", fontsize=11, fontweight='bold', pad=8)
        ax.set_xlabel("Angle (°)", fontsize=9)
        
        # Row label (Mineral Name) on the far left
        if j == 0:
            ax.set_ylabel(f"{SLABELS[struct]}", fontsize=12, fontweight='bold')
        
        ax.set_xlim(xrange[0], xrange[1])
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.15, linestyle=':')

# Global formatting
# fig.suptitle("Bond Angle Distribution Comparison", fontsize=16, fontweight='bold', y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])
# plt.show()

plt.savefig('fig4_bond_distances.png', dpi=300, bbox_inches='tight')

## Radial Distribution Functions

In [ ]:

gr_data = {}
for struct in STRUCTURES:
    for sp_a, sp_b in GR_PAIRS[struct]:
        for model in MODELS:
            r, gr = compute_gr(model, struct, sp_a, sp_b, stride=10)
            gr_data[(model, struct, sp_a, sp_b)] = (r, gr)
            print(f"{MODEL_LABELS[model]:14s} {struct:24s} g({sp_a}-{sp_b}): {'ok' if r is not None else 'missing'}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import savgol_filter

# 1. Setup the Figure with the asymmetrical 6-column grid
num_rows = len(STRUCTURES)
fig = plt.figure(figsize=(12, 3.8 * num_rows))
gs = gridspec.GridSpec(num_rows, 6, hspace=0.45, wspace=0.3)

for i, struct in enumerate(STRUCTURES):
    # Filter out O-O pairs from the GR_PAIRS list
    pairs = [p for p in GR_PAIRS.get(struct, []) if not (p[0] == "O" and p[1] == "O")]
    n_pairs = len(pairs)
    
    if n_pairs == 0:
        continue

    # Asymmetrical Layout Logic (LCM of 2 and 3 is 6)
    if n_pairs == 3:
        width = 2
    else:
        width = 6 // n_pairs 

    for j in range(n_pairs):
        start_col = j * width
        ax = fig.add_subplot(gs[i, start_col : start_col + width])
        
        sp_a, sp_b = pairs[j]

        for model in MODELS:
            r, gr = gr_data.get((model, struct, sp_a, sp_b), (None, None))
            if r is None or len(gr) < 10:
                continue
            
            # --- Savitzky-Golay Smoothing ---
            # window_length: must be odd. Larger = smoother.
            # polyorder: 2 or 3 is usually best to preserve peak shapes.
            try:
                window = 11 if len(gr) > 20 else 5
                gr_smooth = savgol_filter(gr, window_length=window, polyorder=3)
            except:
                gr_smooth = gr # Fallback if data is too short
            
            ax.plot(r, gr_smooth, color=MODEL_COLORS[model], 
                    ls=MODEL_LS[model], lw=2.0, label=MODEL_LABELS[model], alpha=0.9)

        # Legend only on the very first plot (top-left)
        if i == 0 and j == 0:
            ax.legend(fontsize=8, frameon=False, loc='upper right')

        # # Reference line at g(r) = 1
        # ax.axhline(1.0, color="gray", ls="--", lw=0.8, alpha=0.5)

        # Formatting
        ax.set_title(f"g({sp_a}–{sp_b})", fontsize=11, fontweight='bold', pad=8)
        ax.set_xlabel("r (Å)", fontsize=9)
        
        # Row label (Mineral Name) on the far left
        if j == 0:
            ax.set_ylabel(f"{SLABELS[struct]}\ng(r)", fontsize=12, fontweight='bold')
        
        ax.set_xlim(0.5, 6.0)
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.15, linestyle='--')

# Global formatting
# fig.suptitle("Radial Distribution Functions (Smoothed)", fontsize=16, fontweight='bold', y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('fig5_gr.png', dpi=300, bbox_inches='tight')

## Bond Distance Statistics

In [ ]:

stat_rows = []
for struct in STRUCTURES:
    for sp_a, sp_b, cutoff, _ in BOND_PAIRS[struct]:
        for model in MODELS:
            arr = bond_dist_data.get((model, struct, sp_a, sp_b), np.array([]))
            if arr.size:
                stat_rows.append({
                    "model": model,
                    "model_label": MODEL_LABELS[model],
                    "structure": struct,
                    "mineral": SLABELS[struct],
                    "bond": f"{sp_a}-{sp_b}",
                    "mean_a": arr.mean(),
                    "std_a": arr.std(),
                    "n_samples": len(arr),
                })
bond_stats = pd.DataFrame(stat_rows)
bond_stats.sort_values(["structure", "bond", "model"])
